# Part 4 - True-talent xwOBA (empirical Bayes)

This is the part I actually wanted all along: a number for *analyzing a hitter*. For each batter-season I want an xwOBA that's been **regressed for sample size** — a scorching 120-PA line gets pulled back toward the league until the plate appearances justify it — paired with an interval that **narrows as PA grows**.

The method is deliberately simple: Gaussian–Gaussian empirical Bayes, no BART re-fit needed. Each estimate is

`theta_hat = mu + reliability * (raw - mu)`,  where  `reliability = tau^2 / (tau^2 + se^2)`.

In words: start at the league mean `mu`, and move toward the player's raw number only as far as the reliability of their sample allows. Everything here comes from `results/talent/`.

In [ ]:
# --- setup: find repo root, load helpers (run this first) ---
import sys, json
from pathlib import Path
import polars as pl
from IPython.display import Image, Markdown, display

pl.Config.set_tbl_rows(30)
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "config.yaml").exists()), here)
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"

def jload(rel):
    return json.loads((RESULTS / rel).read_text())

def show_fig(rel, width=680):
    p = RESULTS / rel
    return Image(filename=str(p), width=width) if p.exists() else Markdown(f"_missing figure: {rel}_")

print("repo root:", ROOT)
print("results:  ", RESULTS, "(exists)" if RESULTS.exists() else "(MISSING)")

## Per-season hyperparameters

The two knobs — the league mean `mu` and the talent spread `tau` — are fit separately for each season, using only players with 100+ PA, then applied to everyone.

In [ ]:
tm = jload("talent/talent_metrics.json")
pl.DataFrame([
    {"season": r["season"], "n": r["n"], "mu (league)": round(r["mu"], 4),
     "tau (spread)": round(r["tau"], 4), "median reliability": round(r["median_reliability"], 3)}
    for r in tm["per_season"]
])

## Shrinkage in action

Here's the whole idea in one picture: small samples collapse toward the season mean, while big samples barely budge.

In [ ]:
show_fig("talent/figures/shrinkage_raw_to_talent.png")

In [ ]:
show_fig("talent/figures/reliability_vs_pa.png")

## A few concrete examples — raw → talent

Abstractions are easier to trust with names attached, so here are a handful of players run through the estimator.

In [ ]:
t = pl.read_parquet(RESULTS / "talent" / "talent_table.parquet")
def ex(name, season):
    r = t.filter((pl.col("player_name") == name) & (pl.col("season") == season)).row(0, named=True)
    return {"player": name, "season": season, "PA": r["PA"],
            "raw": round(r["xwoba_raw"], 3), "talent": round(r["xwoba_talent"], 3),
            "90% lo": round(r["talent_lo"], 3), "90% hi": round(r["talent_hi"], 3),
            "reliability": round(r["reliability"], 3)}
pl.DataFrame([
    ex("Khalil Lee", 2022),     # 2 PA, extreme hot -> fully regressed to league
    ex("Mike Trout", 2024),     # 125 PA, hot -> pulled down to a believable .341
    ex("Austin Hedges", 2024),  # 143 PA, cold, low-variance -> pulled up
])

Worth pausing on Trout here: he gets regressed *harder* than Hedges at a similar PA count. That's not a bug — it's the estimator leaning on each player's own noisiness (Trout's boom-or-bust, high per-PA variance) rather than just counting plate appearances.

## The payoff — does shrinkage actually predict better?

Fancy method, fine. But does regressing for sample size predict next season any better than the raw number, or than Savant?

In [ ]:
v = tm["validation"]
def row(label, block, has_rmse):
    d = {"population": label, "n": block["n"],
         "EB talent r": round(block["xwoba_talent"]["r"], 4),
         "raw r": round(block["xwoba_raw"]["r"], 4),
         "Savant r": round(block["xwoba_savant"]["r"], 4)}
    return d
pl.DataFrame([
    row("pooled, PA_T >= 100 (vs 0.487 anchor)", v["pooled_pa_min"], True),
    row("pooled, PA_T >= 30 (admits low-PA)", v["pooled_lowpa_inclusive"], False),
])

The verdict: EB talent beats the raw number in both populations, and it **beats Savant once genuinely low-PA seasons are allowed in** (0.467 vs 0.452). At the 100+ PA anchor it ties Savant (0.489 vs 0.491) — the same parity I hit in Part 2, except now the *center* of the estimate is sample-size-honest.

### Why the win shows up pooled, not per-band (and why that isn't a bug)

In [ ]:
bands = pl.DataFrame([
    {"PA band": b["band"], "n": b["n"], "median reliability": round(b["median_reliability"], 2),
     "|talent-raw|": round(b["mean_abs_talent_minus_raw"], 4),
     "talent r": round(b["xwoba_talent"]["r"], 3), "raw r": round(b["xwoba_raw"]["r"], 3),
     "savant r": round(b["xwoba_savant"]["r"], 3)}
    for b in v["by_band"]
])
bands

Here's the subtle part. Inside a narrow PA band, reliability is nearly constant, so `theta_hat = mu(1 - c) + c * raw` is essentially *affine* in the raw number — and Pearson r doesn't care about affine transforms. So per-band r *can't* move much no matter how good the shrinkage is; those tiny per-band gaps are just noise.

Where shrinkage actually pays off is across a *heterogeneous*-PA population: it compresses the wild low-PA raw numbers, and that only shows up in the pooled r. So this is a property of the estimator, not a bug — I wrote it up in `results/talent/NOTES.md`.

## The biggest shrinks

For fun, here are the ten hitters the estimator moved the most — the hot and cold small samples that got yanked back toward the mean.

In [ ]:
pl.DataFrame([
    {"player": r["player_name"], "season": r["season"], "PA": r["PA"],
     "raw": round(r["xwoba_raw"], 3), "talent": round(r["xwoba_talent"], 3),
     "reliability": round(r["reliability"], 3)}
    for r in tm["biggest_shrinks"][:10]
])

## What's next — Phase 2

There's an obvious weakness in all of this: right now every player is shrunk toward the *league* mean. But a rookie who's barreling everything shouldn't regress toward average — he should regress toward a slugger. Phase 2 swaps the flat league prior for a **BART contact-quality prior** that knows this. The plan is written up in `docs/superpowers/plans/2026-07-18-xwobart-phase2-bart-prior.md`.